In [52]:
import os
import requests
import re
import json
import time
from datetime import datetime

with open(".env", "r") as f:
    for line in f: 
        if line.startswith("GITHUB_TOKEN="):
            TOKEN = line.strip().split("=",1)[1]

In [53]:
TOKEN = token.strip().strip("'")

In [54]:
headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json"
}

In [55]:
def parse_pr_link(link):
    pattern = r"https://github\.com/([^/]+)/([^/]+)/pull/(\d+)"
    match = re.match(pattern, link.strip())
    if match:
        owner, repo, pr_number = match.groups()
        return owner, repo, int(pr_number)
    else:
        return None, None, None

In [ ]:
def handle_rate_limit(response):
    if response.status_code == 403 and "X-RateLimit-Remaining" in response.headers:
        remaining = int(response.headers.get("X-RateLimit-Remaining", 0))
        reset_ts = int(response.headers.get("X-RateLimit-Reset", 0))
        now_ts = int(time.time())

        if remaining == 0:
            wait_time = reset_ts - now_ts + 5  
            reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')
            print(f"API limit. Waiting {wait_time}s (reset on {reset_time} UTC)...")
            time.sleep(max(wait_time, 0))
            return True
    return False

In [57]:
with open("pr_links.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

In [58]:
output_path = "reviews_raw.txt"
batch_size = 5000  
total_lines = len(lines)

In [ ]:
print(f"Found {total_lines} link PRs. Batching {batch_size} per part...")

Znaleziono 6618 linków PR. Przetwarzanie po 5000 na partię...


In [ ]:
with open(output_path, "a", encoding="utf-8") as out:
    for i in range(0, total_lines, batch_size):
        batch = lines[i:i + batch_size]
        print(f"\Processing part {i // batch_size + 1} ({len(batch)} links)...")

        for line in batch:
            parts = line.split()
            if len(parts) != 2:
                print(f"Wrong format: {line}")
                continue

            pr_id, link = parts
            owner, repo, pr_number = parse_pr_link(link)
            if not owner:
                print(f"Wrong link: {link}")
                continue

            url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}/reviews"

            while True:
                response = requests.get(url, headers=headers)
                if handle_rate_limit(response):
                    continue

                if response.status_code != 200:
                    print(f"Error {response.status_code} while fetching: {url}")
                    break

                reviews = response.json()
                for review in reviews:
                    user_info = review.get("user")
                    login = user_info.get("login") if user_info else None
                
                    record = {
                        "id": review.get("id"),
                        "pr_id": pr_id,
                        "user": login,
                        "user_type": None,
                        "state": review.get("state"),
                        "submitted_at": review.get("submitted_at"),
                        "body": review.get("body")
                    }
                    out.write(json.dumps(record, ensure_ascii=False) + "\n")
                break 

        if i + batch_size < total_lines:
            print(f"Quick break 10s before part ({i + batch_size}/{total_lines})...")
            time.sleep(10)

print(f"\nCompleted. Data save to: {output_path}")


Przetwarzanie partii 1 (5000 linków)...


/tmp/ipykernel_3688756/134185260.py:9: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')


Limit API osiągnięty. Czekam 826 sek. (reset o 2025-10-22 12:38:50 UTC)...
Krótka przerwa 10 sekund przed kolejną partią (5000/6618)...

Przetwarzanie partii 2 (1618 linków)...
Limit API osiągnięty. Czekam 961 sek. (reset o 2025-10-22 13:38:55 UTC)...

Zakończono. Dane dopisano do pliku: reviews_raw.txt


In [ ]:
# Convert to parquet
import pandas as pd
import json

input_file = "reviews_raw.txt"
output_file = "reviews_data.parquet"

records = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line.strip()))

df = pd.DataFrame(records)
df.to_parquet(output_file, index=False)

print(f"Writing data to parquet file: {output_file}")
print(f"Data count: {len(df)}")

Zapisano dane do pliku Parquet: reviews_data.parquet
Liczba rekordów: 11693


In [ ]:
import pandas as pd

parquet_file = "reviews_data.parquet"
df = pd.read_parquet(parquet_file)

print(df.head(100))

print(f"\Total data count: {len(df)}")

            id       pr_id            user user_type      state  \
0   2618604424  2336888723       ameliahsu      None  COMMENTED   
1   2618606142  2336888723       ameliahsu      None  COMMENTED   
2   2625030950  2336888723      natemoo-re      None  COMMENTED   
3   2625204648  2336888723      natemoo-re      None   APPROVED   
4   2751626124  2447123365  michellewzhang      None   APPROVED   
..         ...         ...             ...       ...        ...   
95  2966928441  2611510009         elie222      None  COMMENTED   
96  2966934770  2611510009         elie222      None  COMMENTED   
97  2966940450  2611510009         elie222      None  COMMENTED   
98  2970654744  2611510009        edulelis      None  COMMENTED   
99  2970668660  2611510009        edulelis      None  COMMENTED   

            submitted_at                                               body  
0   2025-02-14T19:06:37Z                                                     
1   2025-02-14T19:07:42Z               

In [ ]:
unique_values = df['user'].unique()
print(f"Unique values in 'user' column:\n{unique_values}")

value_counts = df['user'].value_counts()
print("\nUnique values counts:")
print(value_counts)


Unikalne wartości w kolumnie 'user':
['ameliahsu' 'natemoo-re' 'michellewzhang' ... 'windsurf-bot[bot]'
 'alaister' 'msukkari']

Liczba wystąpień każdej wartości:
user
coderabbitai[bot]                     678
ellipsis-dev[bot]                     541
copilot-pull-request-reviewer[bot]    445
MH4GF                                 194
lucasgomide                           173
                                     ... 
Lendemor                                1
stc1988                                 1
stormslowly                             1
NicholasLYang                           1
ypatil12                                1
Name: count, Length: 1858, dtype: int64


In [ ]:
bot_rows = df[df['user'].str.endswith('[bot]', na=False)]

unique_bots = bot_rows['user'].unique()
print("Unique values with sufix [bot]:")
print(unique_bots)

print("\nAll rows with sufix [bot]:")
print(bot_rows)

Unikalne wartości z sufiksem [bot]:
['ellipsis-dev[bot]' 'coderabbitai[bot]'
 'copilot-pull-request-reviewer[bot]' 'graphite-app[bot]'
 'greptile-apps[bot]' 'cursor[bot]' 'github-actions[bot]'
 'sourcery-ai[bot]' 'cubic-dev-ai[bot]' 'claude[bot]'
 'gemini-code-assist[bot]' 'github-advanced-security[bot]'
 'qodo-merge-for-open-source[bot]' 'renovate-approve[bot]' 'cdr-bot[bot]'
 'dust-agent[bot]' 'entelligence-ai-pr-reviews[bot]' 'pullrequest[bot]'
 'baz-reviewer[bot]' 'accesslint[bot]' 'codegen-sh[bot]'
 'copilot-swe-agent[bot]' 'mlflow[bot]' 'dotnet-policy-service[bot]'
 'pulumi-renovate-approve[bot]' 'recurseml[bot]' 'standard-input[bot]'
 'callstackai[bot]' 'propel-code-bot[bot]' 'windsurf-bot[bot]']

Wszystkie wiersze z [bot]:
               id       pr_id                user user_type      state  \
38     2600465607  2321189660   ellipsis-dev[bot]       Bot  COMMENTED   
39     2600687573  2321189660   ellipsis-dev[bot]       Bot  COMMENTED   
40     2600696008  2321189660   ellip

In [68]:
df['user_type'] = df['user'].apply(lambda x: "Bot" if isinstance(x, str) and x.endswith('[bot]') else "Human")

In [ ]:
unique_user_type_values = df['user_type'].unique()
print(f"Unique values in 'user_type' column:\n{unique_user_type_values}")

Unikalne wartości w kolumnie 'user_type':
['Human' 'Bot']


In [ ]:
output_file = "human_pull_request_reviews.parquet"

df.to_parquet(output_file, index=False)

print(f"Writing modified DataFrame to Parquet file: {output_file}")

Zapisano zmodyfikowany DataFrame do pliku Parquet: human_pull_request_reviews.parquet
